In [1]:
import os
import pandas as pd

In [2]:
# 파일 목록 생성 (p10v32 ~ p25v32)
file_list = []
for year in range(10, 25):  # 10~25 (16 개)
    filename = f'p{year}v32_KMP_csv.csv'
    file_list.append(filename)

In [3]:
# 연도별 컬럼 생성 함수 (모든 컬럼에 연도 포함)
def get_columns_by_year(year):
    """
    조사 연도에 따른 모든 컬럼 반환
    컬럼 이름의 __에는 조사 연도가 들어간다
    """

    # 모든 컬럼에 연도 포함 (pid 는 연도 무관)
    columns = [
        'pid',                     # 개인 통합 ID (연도 무관)
        f'p{year}age',             # 나이 카테고리
        f'p{year}gender',          # 성별
        f'p{year}income',          # 소득 카테고리
        f'p{year}school',          # 학력
        f'p{year}job1',            # 직업 유무
        f'p{year}mar',             # 결혼
        f'p{year}c01001',          # 월평균 휴대폰 이용 총 금액
        f'p{year}c01003',          # 월평균 기기 할부금
        f'p{year}c02001',          # 휴대폰 요금 부담자
    ]

    # 연도 의존 컬럼 추가
    if year >= 11:
        columns.append(f'p{year}hhldsiz')   # 가구원 수 (11년~24년)

    if year >= 15:
        columns.append(f'p{year}a02014')    # 일반휴대폰 가입 이동 통신사 (15년~24년)
        columns.append(f'p{year}a03008')    # 스마트폰 가입 이동 통신사 (15년~24년)

    if year <= 14:
        columns.append(f'p{year}a01027')    # 가입 이동 통신사 (10년~14년)

    if year >= 17:
        columns.append(f'p{year}c02003')    # 휴대폰 결합상품 가입 여부 (17년~24년)

    return columns

In [4]:
def rename_yeared_columns(df, year):
    """
    df 에서 6 개 컬럼 이름을 연도 형식을 유지하며 새 컬럼명으로 변경
    예를 들어 p15a02014 -> p15cellular_provider

    Args:
        df: pandas DataFrame
        year: 연도 (예: 15)

    Returns:
        컬럼명 변경된 DataFrame
    """
    # 원본 컬럼명 -> 새 컬럼명 매핑 (연도 포함)
    rename_map = {
        "pid": "id",
        f'p{year}age': 'age',
        f'p{year}gender': 'gender',
        f'p{year}income': 'income',
        f'p{year}school': 'school',
        f'p{year}job1': 'job',
        f'p{year}mar': 'marriage',
        f'p{year}hhldsiz': 'household_size',
        f'p{year}a02014': 'cellular_provider',      # 일반휴대폰 가입 이동 통신사
        f'p{year}a03008': 'smartphone_provider',    # 스마트폰 가입 이동 통신사
        f'p{year}a01027': 'mobile_provider',        # 가입 이동 통신사 (10-14 년)
        f'p{year}c01001': 'monthly_total_cost',     # 월평균 휴대폰 이용 총 금액
        f'p{year}c01003': 'monthly_installment',    # 월평균 기기 할부금
        f'p{year}c02001': 'cost_payer',             # 휴대폰 요금 부담자
        f'p{year}c02003': 'is_mobile_bundled',      # 휴대폰 결합상품 가입 여부
    }

    # 파일에 실제로 존재하는 컬럼만 필터링
    existing_rename_map = {k: v for k, v in rename_map.items() if k in df.columns}

    # 컬럼명 변경 (rename 사용)
    df_renamed = df.rename(columns=existing_rename_map)
    df_renamed["year"] = year

    return df_renamed

In [5]:
def convert_str_columns_to_int64(df, year):
    cols = [
        'monthly_total_cost',
        'monthly_installment',
        'cost_payer',
        'mobile_provider',
        'cellular_provider',
        'smartphone_provider',
        'is_mobile_bundled',
    ]
    existing_cols = [c for c in cols if c in df.columns]

    for col in existing_cols:
        df[col] = (
            df[col]
            .replace(["", " ", "nan", "None", "null"], pd.NA)
            .astype("string")
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return df

In [6]:
def replace_9999_to_na(df):
    int_cols = [c for c in df.columns if pd.api.types.is_integer_dtype(df[c]) or pd.api.types.is_numeric_dtype(df[c])]
    for col in int_cols:
        df[col] = df[col].replace(9999, pd.NA)
    return df

In [7]:
def make_provider_column(df, year):
    # 연도에 따른 원본 변수명 후보
    mobile_col = 'mobile_provider'
    smartphone_col = 'smartphone_provider'
    cellular_col = 'cellular_provider'

    # 컬럼이 이미 존재하는지 확인
    has_mobile = mobile_col in df.columns
    has_smartphone = smartphone_col in df.columns
    has_cellular = cellular_col in df.columns

    if year <= 14:
        if has_mobile:
            df['provider'] = df[mobile_col]
        else:
            df['provider'] = pd.NA

    else:
        if has_smartphone and has_cellular:
            df['provider'] = df[smartphone_col].combine_first(df[cellular_col])
        elif has_smartphone:
            df['provider'] = df[smartphone_col]
        elif has_cellular:
            df['provider'] = df[cellular_col]
        else:
            df['provider'] = pd.NA

    for col in ['mobile_provider', 'smartphone_provider', 'cellular_provider']:
        if col in df.columns:
            df = df.drop(columns=col)

    return df

In [8]:
def scale_2010_columns(df):
    target_cols = ['monthly_total_cost', 'monthly_installment']

    for col in target_cols:
        if col in df.columns:
            df[col] = df[col] // 1000

In [9]:
# 모든 파일에서 컬럼 추출
dataframes = []

for filepath in file_list:
    if not os.path.exists(filepath):
        print(f"{filepath} not found")
        continue

    year = int(filepath[1:3])  # 연도 추출
    requested_columns = get_columns_by_year(year)

    try:
        # CSV 읽기 (전체 컬럼 읽음)
        df = pd.read_csv(
            filepath,
            encoding='utf-8',
            low_memory=False,           # DtypeWarning 해결: 파일 한 번에 읽기
            usecols=requested_columns,  # 모든 컬럼을 문자일로 읽어서 mixed types 문제 해결
        )

        # 요청한 컬럼 중 파일에 실제로 존재하는 컬럼만 필터링
        available_columns = [col for col in requested_columns if col in df.columns]
        df = df[available_columns]

        # 컬럼명 변경
        df = rename_yeared_columns(df, year)

        # 컬럼 데이터 타입 변경
        convert_str_columns_to_int64(df, year)

        # 모름/무응답을 결측 처리
        df = replace_9999_to_na(df)

        # 가입 이동 통신사 컬럼을 통합
        df = make_provider_column(df, year)

        dataframes.append(df)
        print(f"{filepath} (연도 {year}): {len(df)}행, {len(df.columns)}컬럼 추출 완료")

    except Exception as e:
        print(f"경고: {filepath} 읽기 실패 - {e}")

# 2010년 데이터 컬럼 단위를 다른 연도와 통일
scale_2010_columns(dataframes[0])

p10v32_KMP_csv.csv (연도 10): 6737행, 12컬럼 추출 완료
p11v32_KMP_csv.csv (연도 11): 12000행, 13컬럼 추출 완료
p12v32_KMP_csv.csv (연도 12): 10319행, 13컬럼 추출 완료
p13v32_KMP_csv.csv (연도 13): 10464행, 13컬럼 추출 완료
p14v32_KMP_csv.csv (연도 14): 10172행, 13컬럼 추출 완료
p15v32_KMP_csv.csv (연도 15): 9873행, 13컬럼 추출 완료
p16v32_KMP_csv.csv (연도 16): 9788행, 13컬럼 추출 완료
p17v32_KMP_csv.csv (연도 17): 9425행, 14컬럼 추출 완료
p18v32_KMP_csv.csv (연도 18): 9426행, 14컬럼 추출 완료
p19v32_KMP_csv.csv (연도 19): 10864행, 14컬럼 추출 완료
p20v32_KMP_csv.csv (연도 20): 10302행, 14컬럼 추출 완료
p21v32_KMP_csv.csv (연도 21): 10154행, 14컬럼 추출 완료
p22v32_KMP_csv.csv (연도 22): 9941행, 14컬럼 추출 완료
p23v32_KMP_csv.csv (연도 23): 9757행, 14컬럼 추출 완료
p24v32_KMP_csv.csv (연도 24): 8693행, 14컬럼 추출 완료


In [10]:
for df in dataframes:
    print()
    print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 6737 entries, 0 to 6736
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   6737 non-null   int64 
 1   age                  6735 non-null   object
 2   gender               6737 non-null   int64 
 3   income               6612 non-null   object
 4   school               6729 non-null   object
 5   job                  6737 non-null   int64 
 6   marriage             6737 non-null   int64 
 7   monthly_total_cost   5853 non-null   Int64 
 8   monthly_installment  4680 non-null   Int64 
 9   cost_payer           5938 non-null   Int64 
 10  year                 6737 non-null   int64 
 11  provider             5921 non-null   Int64 
dtypes: Int64(4), int64(5), object(3)
memory usage: 658.0+ KB
None

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
--

In [11]:
# 데이터프레임마다 컬럼별 값을 확인
for i, df in enumerate(dataframes):
    for col in df.columns:
        if col.endswith('cost_payer'):
            print()
            print(df[col].value_counts(dropna=False).sort_index().head(20))

            # 결측 비율 출력
            na_ratio = df[col].isna().sum() / len(df[col])
            print(f"NA 비율: {na_ratio:.4f} ({na_ratio * 100:.2f}%)")


cost_payer
1       3802
2         51
3         18
4       1850
5        189
6         28
<NA>     799
Name: count, dtype: Int64
NA 비율: 0.1186 (11.86%)

cost_payer
1       6402
2         99
3         27
4       3817
5         46
6          6
<NA>    1603
Name: count, dtype: Int64
NA 비율: 0.1336 (13.36%)

cost_payer
1       5687
2         36
3         22
4       3322
5         30
6          1
<NA>    1221
Name: count, dtype: Int64
NA 비율: 0.1183 (11.83%)

cost_payer
1       5925
2         48
3         52
4       2479
5        806
6          1
<NA>    1153
Name: count, dtype: Int64
NA 비율: 0.1102 (11.02%)

cost_payer
1       5905
2         22
3         21
4       2538
5        680
6          2
<NA>    1004
Name: count, dtype: Int64
NA 비율: 0.0987 (9.87%)

cost_payer
1       5934
2         39
3         28
4       2210
5        832
<NA>     830
Name: count, dtype: Int64
NA 비율: 0.0841 (8.41%)

cost_payer
1       5852
2         29
3         28
4       2221
5        939
6          2
<NA>     717


In [12]:
# 데이터프레임 합치기
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True).sort_values(['id', 'year'])

    # year 컬럼을 id 컬럼 뒤로 재배치
    cols = combined_df.columns.tolist()
    cols.insert(cols.index('id') + 1, cols.pop(cols.index('year')))
    combined_df = combined_df[cols]

    # 이탈 예측 라벨 컬럼 생성
    next_provider = combined_df.groupby('id')['provider'].shift(-1)
    combined_df['churn'] = (combined_df['provider'] != next_provider).astype('float')

    # 마지막 연도는 다음 해가 없으므로 라벨 제외
    combined_df.loc[next_provider.isna(), 'churn'] = pd.NA

    print(f"\n총 합계: {len(combined_df)}행, {len(combined_df.columns)}컬럼")

    # 결과 저장
    output_filename = "extracted_data.csv"
    combined_df.to_csv(output_filename, encoding='cp949', index=False)
    print(f"결과 저장: {output_filename}")
    print(f"최종 컬럼: {combined_df.columns.tolist()}")
else:
    print("데이터 추출에 실패했습니다.")


총 합계: 147915행, 15컬럼
결과 저장: extracted_data.csv
최종 컬럼: ['id', 'year', 'age', 'gender', 'income', 'school', 'job', 'marriage', 'monthly_total_cost', 'monthly_installment', 'cost_payer', 'provider', 'household_size', 'is_mobile_bundled', 'churn']
